In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping
import pickle
import datetime

c:\DeepLearning_&Regression_ChurnDataset\churn\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [22]:
#load the dataset
df=pd.read_csv("chrun.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [23]:
#preprocesss the data
#drop irrelevant colummns
df=df.drop(['CustomerId','RowNumber',"Surname"],axis=1)

In [4]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [24]:
#encode categroical variable
lable_encoder_gender=LabelEncoder()
df['Gender']=lable_encoder_gender.fit_transform(df['Gender'])

In [25]:
##ohe on geography
ohe=OneHotEncoder()
geo_encoder=ohe.fit_transform(df[['Geography']])
ohe.get_feature_names_out(['Geography'])
geo_encoder_df = pd.DataFrame(
    geo_encoder.toarray(),
    columns=ohe.get_feature_names_out(['Geography']))

In [26]:
#combine ohe with original data
df=pd.concat([df.drop('Geography', axis=1), geo_encoder_df], axis=1)

In [27]:
#save encoder and scaler
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(lable_encoder_gender,file)
with open('geo_encoder_df.pkl','wb') as file:
    pickle.dump(ohe,file)

In [9]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [28]:
#divide the df in independent and depeendent
X=df.drop(['Exited'],axis=1)
y=df['Exited']

In [29]:
#train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42)

In [12]:
print(type(X_train))
print(type(X_test))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>


In [30]:
#scale the features
scaler=StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#also make a pickle file
with open("scaler.pkl",'wb') as file:
    pickle.dump(scaler,file)

In [14]:
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import datetime
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [15]:
#Build and ANN models
model=Sequential(
    [Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
     Dense(32,activation='relu'),
     Dense(1,activation='sigmoid')]
)
model.summary()

c:\DeepLearning_&Regression_ChurnDataset\churn\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

### 🔷 Model Structure
Type: Sequential model
Flow: Input → 64 neurons → 32 neurons → 1 neuron
Used for: Binary classification (yes/no output)
### 🔷 Layers
1. First Layer (dense)
Type: Dense (fully connected)
Output shape: (None, 64)
Meaning:
64 neurons
None = any batch size
Params: 832
Calculation:
(input features × 64) + 64 bias
Input features = 13
2. Second Layer (dense_1)
Output shape: (None, 32)
Meaning:
32 neurons
Params: 2080
Calculation:
(64 × 32) + 32
3. Output Layer (dense_2)
Output shape: (None, 1)
Meaning:
1 neuron → gives final prediction
Params: 33
Calculation:
(32 × 1) + 1
### 🔷 Parameters (Important)
Total params: 2945
Total learnable values (weights + biases)
Trainable params: 2945
All parameters are updated during training
Non-trainable params: 0
Nothing is frozen
### 🔷 Key Concepts
Dense layer:
Every neuron connects to all neurons in previous layer
Output shape:
Shows number of neurons per layer
Parameters:
Weights + biases learned during training

In [16]:
#complie the model
opt =tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=opt,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [17]:
#Setup the tensorboard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1) #“Record histograms of weights (and optionally activations) every 1 epoch.” if set to 5 then every 5 s+epoch

In [18]:
#setup early stopping...wait for 10 epochs if no chnage then stop training th emodel and it is on unseen data validation loss
early_stopping_callback=EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

In [19]:
#train the model {{{{learningggggg=model.fit()}}}} & the name is "history" because it store the data for training
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=32,
    callbacks=[tensorboard_callback, early_stopping_callback]
)
#validation_loss=used to check the data after epoch and epoch=100 but not then early_stopping_callback if pateince meaning if 10 checked then if no improvement then stop after 10

Epoch 1/100


TypeError: Descriptors cannot be created directly.
If this call came from a _pb2.py file, your generated code is out of date and must be regenerated with protoc >= 3.19.0.
If you cannot immediately regenerate your protos, some other possible workarounds are:
 1. Downgrade the protobuf package to 3.20.x or lower.
 2. Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python (but this will use pure-Python parsing and will be much slower).

More information: https://developers.google.com/protocol-buffers/docs/news/2022-05-06#python-updates

In [ ]:
#save the model file
model.save("model.h5")

In [ ]:
#load tenssorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [ ]:
import os

# Check if logs exist
if not os.path.exists('logs/fit') or not os.listdir('logs/fit'):
    print("⚠️  No training logs found. Run model.fit() first!")
else:
    print(f"✅ Found training logs: {os.listdir('logs/fit')}")
    
    # Try using the TensorBoard program API
    try:
        from tensorboard import program
        
        tb = program.TensorBoard()
        tb.configure(argv=['tensorboard', '--logdir', os.path.abspath('logs/fit'), '--port', '6006'])
        url = tb.launch()
        print(f"✅ TensorBoard started at: {url}")
    except Exception as e:
        print(f"Could not use TensorBoard API: {e}")
        print("\n📌 Alternative: Open http://localhost:6006 in your browser (may take 10 seconds to load)")

ModuleNotFoundError: No module named 'matplotlib'